In [ ]:
!git clone https://github.com/mmarcotullio/KernelOracle.git
%cd KernelOracle
!git checkout mansheel-updates
!ls

Cloning into 'KernelOracle'...
remote: Enumerating objects: 990, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 990 (delta 37), reused 17 (delta 3), pack-reused 861 (from 2)
Receiving objects: 100% (990/990), 79.44 MiB | 33.87 MiB/s, done.
Resolving deltas: 100% (227/227), done.
Filtering content: 100% (10/10), 4.20 GiB | 35.02 MiB/s, done.
Encountered 1 file(s) that should have been pointers, but weren't:
	TCN/data/collect_traces_docs.txt
/content/KernelOracle
Branch 'mansheel-updates' set up to track remote branch 'mansheel-updates' from 'origin'.
Switched to a new branch 'mansheel-updates'
 data								  predict3.pdf
 data_preprocessing.py						  predict4.pdf
'Deep task scheduler for the Linux kernel - Project report.pdf'   predict5.pdf
 LICENSE							  predict6.pdf
 lkp_project.yml						  predict7.pdf
 model.py							  predict8.pdf
 predict0.pdf							  predict9.pdf
 predict10.pdf							  README.md
 predict11.p

In [ ]:
!pip -q install torch pandas numpy scikit-learn matplotlib tqdm

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

DATA_DIR = "/content/drive/MyDrive/data_tcn"
print("Files in DATA_DIR:", os.listdir(DATA_DIR))

Files in DATA_DIR: ['all.csv', 'test_seen.csv', 'test_unseen.csv', 'train.csv']


In [ ]:
!ln -s "/content/drive/MyDrive/data_tcn" data_tcn
!ls -lh data_tcn
!head -n 2 data_tcn/train.csv

lrwxrwxrwx 1 root root 31 Mar  2 20:06 data_tcn -> /content/drive/MyDrive/data_tcn
timestamp,time_diff,task_name,pid,cpu,prev_state,prev_comm,workload_type,trace_id
3710767.341761,0.000000000,code,861190,003,R,swapper/3,cpu_bound_4p,cpu_bound_4p_run1


In [ ]:
import numpy as np
import json
import os


def build_vocab(df, vocab_out_path):



    df["pid_str"] = df["pid"].astype(str)

    unique_pids = sorted(df["pid_str"].unique())


    pid_to_idx = {"<UNK>": 0}
    for i, pid in enumerate(unique_pids, start=1):
        pid_to_idx[pid] = i


    unique_states = sorted(df["prev_state"].astype(str).unique())
    state_to_idx = {state: i for i, state in enumerate(unique_states)}

    vocab = {
        "pid_to_idx": pid_to_idx,
        "state_to_idx": state_to_idx,
    }

    with open(vocab_out_path, "w") as f:
        json.dump(vocab, f)

    return vocab



def load_vocab(vocab_path):
    with open(vocab_path, "r") as f:
        return json.load(f)



def encode_df(df, vocab):


    df["pid_str"] = df["pid"].astype(str)

    pid_to_idx = vocab["pid_to_idx"]

    if "<UNK>" not in pid_to_idx:
        pid_to_idx["<UNK>"] = 0

    unk_idx = pid_to_idx["<UNK>"]


    df["pid_idx"] = (
        df["pid_str"]
        .map(lambda x: pid_to_idx.get(x, unk_idx))
        .astype(np.int64)
    )


    state_to_idx = vocab["state_to_idx"]
    df["state_idx"] = (
        df["prev_state"].astype(str)
        .map(lambda x: state_to_idx.get(x, 0))
        .astype(np.int64)
    )

    return df

In [ ]:
rm -rf tcn/artifacts/*

In [ ]:
%%bash

FILE="tcn/scripts/preprocess.py"


sed -i 's/df\["pid_idx"\] = df\["pid_str"\].map(vocab\["pid_to_idx"\]).astype(np.int64)/pid_to_idx = vocab["pid_to_idx"]\n    if "<UNK>" not in pid_to_idx:\n        pid_to_idx["<UNK>"] = 0\n    unk_idx = pid_to_idx["<UNK>"]\n    df["pid_idx"] = df["pid_str"].map(lambda x: pid_to_idx.get(x, unk_idx)).astype(np.int64)/' $FILE

echo "Patched preprocess.py"

Patched preprocess.py


In [ ]:
!sed -n '35,65p' tcn/scripts/preprocess.py

        "pid_to_idx": pid_to_idx,
        "idx_to_pid": idx_to_pid,
        "state_to_idx": state_to_idx,
        "idx_to_state": idx_to_state,
    }


def encode_df(df: pd.DataFrame, vocab: Dict) -> pd.DataFrame:
    df = df.copy()

    df["pid_str"] = df["pid"].astype(str)
    pid_to_idx = vocab["pid_to_idx"]
    if "<UNK>" not in pid_to_idx:
        pid_to_idx["<UNK>"] = 0
    unk_idx = pid_to_idx["<UNK>"]
    df["pid_idx"] = df["pid_str"].map(lambda x: pid_to_idx.get(x, unk_idx)).astype(np.int64)

    df["prev_state_str"] = df["prev_state"].astype(str).fillna("UNK")

    if "UNK" not in vocab["state_to_idx"]:

        vocab["state_to_idx"]["UNK"] = len(vocab["state_to_idx"])
        vocab["idx_to_state"][vocab["state_to_idx"]["UNK"]] = "UNK"

    df["state_idx"] = df["prev_state_str"].map(lambda s: vocab["state_to_idx"].get(s, vocab["state_to_idx"]["UNK"])).astype(np.int64)


    df["cpu"] = pd.to_numeric(df.get("cpu", 0), errors="coerce").fillna(0.0).astype(np.float32)
    df["tim

In [ ]:
!rm -rf tcn/artifacts/*

In [ ]:
# Train
!python tcn/scripts/preprocess.py \
 --csv data_tcn/train.csv \
 --out tcn/artifacts/train.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 1

# Seen
!python tcn/scripts/preprocess.py \
 --csv data_tcn/test_seen.csv \
 --out tcn/artifacts/test_seen.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 1 \
 --use_existing_vocab

# Unseen
!python tcn/scripts/preprocess.py \
 --csv data_tcn/test_unseen.csv \
 --out tcn/artifacts/test_unseen.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 1 \
 --use_existing_vocab

^C
Saved tcn/artifacts/test_seen.npz
Windows: 3169852, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json
^C


In [ ]:
ls -lh tcn/artifacts

total 41M
-rw-r--r-- 1 root root  41M Mar  2 20:21 test_seen.npz
-rw-r--r-- 1 root root 124K Mar  2 20:19 vocab.json


In [ ]:
!rm -rf tcn/artifacts/*
!ls -lh tcn/artifacts

total 0


In [ ]:
!rm -rf tcn/artifacts/*

In [ ]:
!python tcn/scripts/preprocess.py \
 --csv data_tcn/train.csv \
 --out tcn/artifacts/train.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 4

Saved tcn/artifacts/train.npz
Windows: 1812164, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json


In [ ]:
!python tcn/scripts/preprocess.py \
 --csv data_tcn/test_seen.csv \
 --out tcn/artifacts/test_seen.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 4 \
 --use_existing_vocab

Saved tcn/artifacts/test_seen.npz
Windows: 792463, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json


In [ ]:
!python tcn/scripts/preprocess.py \
 --csv data_tcn/test_unseen.csv \
 --out tcn/artifacts/test_unseen.npz \
 --vocab_out tcn/artifacts/vocab.json \
 --seq_len 64 \
 --stride 4 \
 --use_existing_vocab

Saved tcn/artifacts/test_unseen.npz
Windows: 2172560, seq_len=64, cont_dim=2
Num pids: 3803, Num states: 8
Vocab at: tcn/artifacts/vocab.json


In [ ]:
!ls -lh tcn/artifacts

total 103M
-rw-r--r-- 1 root root  16M Mar  2 20:35 test_seen.npz
-rw-r--r-- 1 root root  38M Mar  2 20:37 test_unseen.npz
-rw-r--r-- 1 root root  51M Mar  2 20:34 train.npz
-rw-r--r-- 1 root root 124K Mar  2 20:33 vocab.json


In [ ]:
import sys
sys.path.append("/content/KernelOracle")

In [ ]:
%cd /content/KernelOracle

/content/KernelOracle


In [ ]:
!python tcn/train_tcn.py \
  --train_npz tcn/artifacts/train.npz \
  --val_npz tcn/artifacts/test_seen.npz \
  --vocab tcn/artifacts/vocab.json \
  --epochs 1 \
  --batch_size 128 \
  --lr 0.001 \
  --device cuda

Traceback (most recent call last):
  File "/content/KernelOracle/tcn/train_tcn.py", line 21, in <module>
    from tcn.models.tcn import TCNConfig, TCNNextPid
ModuleNotFoundError: No module named 'tcn'


In [ ]:
%cd /content/KernelOracle
!git branch

/content/KernelOracle
  main
* mansheel-updates


In [ ]:
%cd /content/KernelOracle

!python -m tcn.train_tcn \
  --train_npz tcn/artifacts/train.npz \
  --test_seen_npz tcn/artifacts/test_seen.npz \
  --test_unseen_npz tcn/artifacts/test_unseen.npz \
  --vocab tcn/artifacts/vocab.json \
  --epochs 1 \
  --batch_size 128 \
  --lr 0.001 \
  --save_dir tcn/checkpoints

/content/KernelOracle
epoch=1 loss=4.6549 time=17935.1s | seen_acc=0.9117 | unseen_acc=0.9835
Saved best checkpoint -> tcn/checkpoints/tcn_nextpid_best.pt
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/KernelOracle/tcn/train_tcn.py", line 149, in <module>
    main()
  File "/content/KernelOracle/tcn/train_tcn.py", line 139, in main
    lat_ms = measure_inference_latency_ms(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/content/KernelOracle/tcn/utils/metrics.py", line 38, in measure_inference_latency_ms
    _ = model(batch["pid"], batch["cont"])
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    ret

In [ ]:
%%bash
sed -i 's/model(batch\["pid"\], batch\["cont"\])/model(batch["pid"], batch["cont"], state=batch["state"])/' tcn/utils/metrics.py
echo "Latency function patched."

Latency function patched.


In [ ]:
import torch
from tcn.models.tcn import TCNConfig, TCNNextPid
from tcn.utils.data import TraceWindowDataset, Vocab, batch_to_device
from tcn.utils.metrics import measure_inference_latency_ms
from torch.utils.data import DataLoader


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


vocab = Vocab.load("tcn/artifacts/vocab.json")


ckpt = torch.load(
    "tcn/checkpoints/tcn_nextpid_best.pt",
    map_location=device
)


cfg = TCNConfig(**ckpt["cfg"])
model = TCNNextPid(cfg).to(device)
model.load_state_dict(ckpt["state_dict"])
model.eval()


ds = TraceWindowDataset("tcn/artifacts/train.npz")
loader = DataLoader(ds, batch_size=128, shuffle=False)

batch = next(iter(loader))
batch = batch_to_device(batch, device)


lat_ms = measure_inference_latency_ms(
    model,
    {"pid": batch["pid"], "cont": batch["cont"], "state": batch["state"]},
    warmup=20,
    iters=100
)

print(f"\nAvg forward latency per batch: {lat_ms:.3f} ms ({device})")


print(f"Latency per sample: {(lat_ms/128):.6f} ms")

Using device: cpu

Avg forward latency per batch: 293.466 ms (cpu)
Latency per sample: 2.292701 ms
